In [ ]:
"""
Stage 6 — Backtesting Engine
Stock Return Prediction | Quant Research Roadmap — Tier 2

Three layers of increasing realism:

  Layer 1 — Naive backtest         : signal × return, flat position sizing
  Layer 2 — Realistic backtest     : transaction costs + slippage + signal threshold
  Layer 3 — Kelly backtest         : volatility-scaled position sizing (Kelly Criterion)

Walk-forward validation:
  Rolls a training window forward in 6-month steps.
  Retrains the model on each fold to prevent the model from using
  a parameter set tuned on data it never should have seen.

Outputs:
  backtest/TICKER_naive.csv
  backtest/TICKER_realistic.csv
  backtest/TICKER_kelly.csv
  backtest/TICKER_walkforward.csv
  backtest/summary.csv

Key realistic frictions modelled:
  - Bid-ask spread        : 0.05% per trade (NSE liquid large-caps)
  - Market impact         : 0.03% × sqrt(order_size / avg_volume)
  - Brokerage commission  : 0.03% per trade (Zerodha/discount broker)
  - STT (Securities Tx Tax): 0.025% on sell side (Indian equity specific)
  - SEBI charges          : 0.0001% per trade
  - Slippage              : 0.02% assumed execution slippage
  Total one-way cost estimate: ~0.13–0.15% per trade

Position limits:
  - Max position size     : 100% of capital (no leverage at fresher stage)
  - Min signal threshold  : 0.0 (trade on any non-zero prediction)
  - Kelly fraction        : half-Kelly (full Kelly is too aggressive)
"""

import os
import sys
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

sys.path.append(os.path.dirname(os.path.abspath(__file__)))
from main import load_sequences, SEQ_DIR, NORM_COLS, WINDOW

BACKTEST_DIR  = "backtest"
MODEL_DIR     = "models"
RESULTS_DIR   = "results"

TICKERS = [
    "RELIANCE", "TCS", "HDFCBANK", "INFY", "ICICIBANK",
    "HINDUNILVR", "ITC", "SBIN", "BHARTIARTL", "KOTAKBANK",
    "LT", "AXISBANK", "ASIANPAINT", "MARUTI", "TITAN",
]

TRADING_DAYS  = 252
RISK_FREE     = 0.065
RF_DAILY      = RISK_FREE / TRADING_DAYS

# ── Realistic NSE transaction cost breakdown ──────────────────────────────────
# These are real numbers for Indian equity markets (2024)
BID_ASK_SPREAD    = 0.0005    # 0.05% — large-cap NSE stocks
BROKERAGE         = 0.0003    # 0.03% — discount broker (Zerodha flat rate)
STT_SELL          = 0.00025   # 0.025% — Securities Transaction Tax (sell only)
SEBI_CHARGE       = 0.000001  # 0.0001%
SLIPPAGE          = 0.0002    # 0.02% — execution slippage estimate
MARKET_IMPACT_K   = 0.0003    # scales with sqrt(order/avg_volume)

# Total one-way cost (excluding market impact which is position-size dependent)
BASE_ONE_WAY_COST = BID_ASK_SPREAD / 2 + BROKERAGE + SEBI_CHARGE + SLIPPAGE
# STT only on sells → added separately on exit

HALF_KELLY        = 0.5       # fractional Kelly — conservative
MAX_POSITION      = 1.0       # cap at 100% of capital (no leverage)
MIN_POSITION      = 0.05      # ignore Kelly sizes below 5% (noise)
SIGNAL_THRESHOLD  = 0.0       # minimum |predicted return| to trade


# ═══════════════════════════════════════════════════════════════════════════════
# TRANSACTION COST MODEL
# ═══════════════════════════════════════════════════════════════════════════════

def transaction_cost(position_change: float,
                     is_sell: bool,
                     avg_volume_ratio: float = 1.0) -> float:
    """
    Computes realistic total transaction cost for one position change.

    position_change      : size of change (0.0 to 1.0 of capital)
    is_sell              : True if closing a long or opening a short
    avg_volume_ratio     : order_size / avg_daily_volume (default 1x = small order)

    Market impact grows with sqrt(order_size / volume) — the Almgren-Chriss
    approximation. For a small retail/prop account trading Nifty50 large-caps,
    this is negligible. Include it anyway so the model is deployable at scale.

    Returns total cost as a fraction of position value.
    """
    if abs(position_change) < 1e-6:
        return 0.0

    base_cost    = BASE_ONE_WAY_COST * abs(position_change)
    stt          = STT_SELL * abs(position_change) if is_sell else 0.0
    mkt_impact   = MARKET_IMPACT_K * np.sqrt(abs(position_change) * avg_volume_ratio)

    return base_cost + stt + mkt_impact


# ═══════════════════════════════════════════════════════════════════════════════
# LAYER 1 — NAIVE BACKTEST
# ═══════════════════════════════════════════════════════════════════════════════

def naive_backtest(y_true: np.ndarray,
                   y_pred: np.ndarray,
                   dates:  pd.DatetimeIndex) -> pd.DataFrame:
    """
    Simplest possible backtest.
    Position = sign of prediction: +1 long, -1 short, no costs.

    Purpose: establishes the upper bound on strategy performance.
    Any realistic backtest must score below this.
    If your realistic backtest beats your naive one → bug in cost model.

    This is the version most beginners show in portfolios.
    The gap between naive and realistic is what interviewers test you on.
    """
    positions   = np.sign(y_pred)
    strategy    = positions * y_true
    buyhold     = y_true.copy()

    df = pd.DataFrame({
        "actual"           : y_true,
        "predicted"        : y_pred,
        "position"         : positions,
        "strategy_ret"     : strategy,
        "buyhold_ret"      : buyhold,
    }, index=dates)

    df["strategy_equity"] = (1 + df["strategy_ret"]).cumprod()
    df["buyhold_equity"]  = (1 + df["buyhold_ret"]).cumprod()
    return df


# ═══════════════════════════════════════════════════════════════════════════════
# LAYER 2 — REALISTIC BACKTEST
# ═══════════════════════════════════════════════════════════════════════════════

def realistic_backtest(y_true:    np.ndarray,
                       y_pred:    np.ndarray,
                       dates:     pd.DatetimeIndex,
                       threshold: float = SIGNAL_THRESHOLD) -> pd.DataFrame:
    """
    Adds three real-world frictions the naive backtest ignores:

    1. Signal threshold
       Only trade when |predicted_return| > threshold.
       Avoids chasing tiny predicted moves that are within noise.
       Reduces turnover (number of trades) substantially.

    2. Realistic transaction costs
       Bid-ask, brokerage, STT, SEBI charges, slippage, market impact.
       Applied on position CHANGES only — holding a position costs nothing.

    3. Position persistence
       Track the current held position explicitly.
       If position doesn't change (model stays long two days in a row),
       no cost is charged on day 2.

    Day-by-day loop instead of vectorised operations because:
      - We need to track the running position explicitly
      - Cost on entry differs from cost on exit (STT on sell side only)
      - Makes the logic auditable — you can explain each line to an interviewer
    """
    n           = len(y_true)
    records     = []
    position    = 0.0      # current held position: +1, -1, or 0
    capital     = 1.0      # normalised starting capital

    for i in range(n):
        pred   = y_pred[i]
        actual = y_true[i]

        # Determine desired position
        if pred > threshold:
            desired = 1.0
        elif pred < -threshold:
            desired = -1.0
        else:
            desired = 0.0

        # Compute position change and associated costs
        delta    = desired - position
        is_sell  = (delta < 0)   # reducing a long or adding a short
        cost     = transaction_cost(delta, is_sell)

        # P&L this period
        gross_ret  = position * actual      # return from held position
        net_ret    = gross_ret - cost       # after friction

        # Update capital
        capital   *= (1 + net_ret)
        position   = desired

        records.append({
            "actual"        : actual,
            "predicted"     : pred,
            "desired_pos"   : desired,
            "position"      : position,
            "delta"         : delta,
            "cost"          : cost,
            "gross_ret"     : gross_ret,
            "net_ret"       : net_ret,
            "capital"       : capital,
        })

    df = pd.DataFrame(records, index=dates)
    df["strategy_equity"] = (1 + df["net_ret"]).cumprod()
    df["buyhold_equity"]  = (1 + y_true).cumprod()
    df["buyhold_ret"]     = y_true
    df["n_trades"]        = (df["delta"].abs() > 0.01).cumsum()
    return df


# ═══════════════════════════════════════════════════════════════════════════════
# LAYER 3 — KELLY CRITERION BACKTEST
# ═══════════════════════════════════════════════════════════════════════════════

def kelly_fraction(pred_return: float,
                   rolling_vol:  float,
                   rolling_win_rate: float) -> float:
    """
    Kelly Criterion: size the position proportional to your edge.

    Full Kelly formula for a continuous return distribution:
        f* = μ / σ²

    where:
        μ = expected return (we use predicted return as proxy)
        σ² = variance (we use rolling 21-day realised vol)

    We use HALF-Kelly (f* × 0.5) because:
        - Full Kelly maximises long-run geometric growth but is extremely
          aggressive — a bad run of predictions can blow up capital fast
        - Half-Kelly sacrifices ~25% of max growth but halves variance
        - Standard practice at most systematic funds

    Position is additionally capped at MAX_POSITION (100% of capital)
    and zeroed below MIN_POSITION (5%) to avoid tiny noisy trades.
    """
    if rolling_vol < 1e-6:
        return 0.0

    sigma_sq    = rolling_vol ** 2
    full_kelly  = pred_return / sigma_sq
    half_kelly  = full_kelly * HALF_KELLY

    # Clip to [−MAX_POSITION, +MAX_POSITION]
    sized       = np.clip(half_kelly, -MAX_POSITION, MAX_POSITION)

    # Zero out positions below minimum threshold (noise suppression)
    if abs(sized) < MIN_POSITION:
        return 0.0

    return float(sized)


def kelly_backtest(y_true:    np.ndarray,
                   y_pred:    np.ndarray,
                   dates:     pd.DatetimeIndex,
                   vol_window: int = 21) -> pd.DataFrame:
    """
    Full Kelly-sized backtest with realistic transaction costs.

    Position sizing evolves daily:
        f_t = kelly_fraction(pred_t, realised_vol_{t-1})

    Realised vol is computed on a rolling window of PAST returns only.
    Using future vol to size positions is a form of lookahead bias.

    The key difference from Layer 2:
        Layer 2: binary positions (−1, 0, +1)
        Layer 3: continuous positions (e.g. 0.37, −0.22, 0.61)

    Continuous sizing means:
        - Large predicted moves → larger positions
        - Uncertain/small signals → smaller exposure
        - Naturally reduces turnover on weak signals
    """
    n         = len(y_true)
    records   = []
    position  = 0.0
    capital   = 1.0

    # Initialise rolling return buffer for vol estimation
    ret_buffer = list(y_true[:vol_window])

    for i in range(vol_window, n):
        pred   = y_pred[i]
        actual = y_true[i]

        # Rolling realised vol (past only — no lookahead)
        rolling_vol = np.std(ret_buffer[-vol_window:]) * np.sqrt(TRADING_DAYS)

        # Kelly-sized desired position
        desired  = kelly_fraction(pred, rolling_vol, rolling_vol)

        # Transaction cost on position change
        delta    = desired - position
        is_sell  = (delta < 0)
        cost     = transaction_cost(delta, is_sell)

        # P&L
        gross_ret = position * actual
        net_ret   = gross_ret - cost
        capital  *= (1 + net_ret)
        position  = desired

        # Update rolling buffer
        ret_buffer.append(actual)

        records.append({
            "actual"        : actual,
            "predicted"     : pred,
            "rolling_vol"   : rolling_vol,
            "desired_pos"   : desired,
            "position"      : position,
            "delta"         : delta,
            "cost"          : cost,
            "gross_ret"     : gross_ret,
            "net_ret"       : net_ret,
            "capital"       : capital,
        })

    df = pd.DataFrame(records, index=dates[vol_window:])
    df["strategy_equity"] = (1 + df["net_ret"]).cumprod()
    df["buyhold_equity"]  = (1 + y_true[vol_window:]).cumprod()
    df["buyhold_ret"]     = y_true[vol_window:]
    df["n_trades"]        = (df["delta"].abs() > 0.01).cumsum()
    return df


# ═══════════════════════════════════════════════════════════════════════════════
# WALK-FORWARD VALIDATION
# ═══════════════════════════════════════════════════════════════════════════════

def walk_forward_backtest(y_true:       np.ndarray,
                          y_pred:       np.ndarray,
                          dates:        pd.DatetimeIndex,
                          train_window: int = 504,   # 2 years
                          test_window:  int = 126,   # 6 months
                          step:         int = 63     # 3-month step
                          ) -> pd.DataFrame:
    """
    Walk-forward validation — the gold standard for time-series backtesting.

    Instead of one fixed train/test split, we roll a window forward:

        Fold 1: Train [0:504]    → Test [504:630]
        Fold 2: Train [63:567]   → Test [567:693]
        Fold 3: Train [126:630]  → Test [630:756]
        ...

    Why this matters:
        A single train/test split might get lucky or unlucky depending
        on which market regime falls in the test period.
        Walk-forward averages over multiple regimes — bull, bear, sideways —
        giving a more reliable estimate of live performance.

    IMPORTANT: We use the ALREADY TRAINED model's predictions (y_pred) here.
        A full walk-forward would retrain the LSTM on each fold, which is
        computationally expensive. For this project, we slice the predictions
        into walk-forward folds to evaluate consistency across time periods.
        This is standard practice for fresher-level quant projects.

    For production, you'd retrain on each fold (see note in output).
    """
    n       = len(y_true)
    results = []
    fold    = 0

    start = train_window    # first test window starts after initial train window

    while start + test_window <= n:
        test_slice  = slice(start, start + test_window)
        fold_true   = y_true[test_slice]
        fold_pred   = y_pred[test_slice]
        fold_dates  = dates[test_slice]

        # Run realistic backtest on this fold
        fold_df     = realistic_backtest(fold_true, fold_pred, fold_dates)
        fold_rets   = fold_df["net_ret"].values

        # Compute metrics for this fold
        ann_factor  = TRADING_DAYS / len(fold_rets)
        ann_ret     = float((1 + fold_rets).prod() ** ann_factor - 1)
        sharpe      = float(
            (fold_rets.mean() - RF_DAILY) / fold_rets.std() * np.sqrt(TRADING_DAYS)
        ) if fold_rets.std() > 0 else 0.0
        dir_acc     = float(((fold_pred > 0) == (fold_true > 0)).mean())
        ic, _       = stats.spearmanr(fold_pred, fold_true)
        eq          = (1 + fold_rets).cumprod()
        mdd         = float(((eq - np.maximum.accumulate(eq)) / np.maximum.accumulate(eq)).min())
        n_trades    = int((fold_df["delta"].abs() > 0.01).sum())

        results.append({
            "fold"          : fold,
            "test_start"    : fold_dates[0].date(),
            "test_end"      : fold_dates[-1].date(),
            "n_days"        : len(fold_rets),
            "ann_return"    : ann_ret,
            "sharpe"        : sharpe,
            "dir_accuracy"  : dir_acc,
            "ic"            : float(ic),
            "max_drawdown"  : mdd,
            "n_trades"      : n_trades,
            "total_cost"    : float(fold_df["cost"].sum()),
        })

        fold  += 1
        start += step

    summary = pd.DataFrame(results)
    return summary


# ═══════════════════════════════════════════════════════════════════════════════
# PERFORMANCE METRICS (shared across all layers)
# ═══════════════════════════════════════════════════════════════════════════════

def compute_metrics(df: pd.DataFrame,
                    ret_col: str = "net_ret",
                    equity_col: str = "strategy_equity") -> dict:
    """Computes full metric suite for any backtest DataFrame."""
    rets      = df[ret_col].values
    equity    = df[equity_col].values
    n         = len(rets)

    ann_ret   = float((1 + rets).prod() ** (TRADING_DAYS / n) - 1)
    excess    = rets - RF_DAILY
    sharpe    = float(excess.mean() / excess.std() * np.sqrt(TRADING_DAYS)) \
                if excess.std() > 0 else 0.0
    downside  = excess[excess < 0]
    sortino   = float(excess.mean() / downside.std() * np.sqrt(TRADING_DAYS)) \
                if len(downside) > 0 and downside.std() > 0 else np.nan
    roll_max  = np.maximum.accumulate(equity)
    drawdowns = (equity - roll_max) / roll_max
    mdd       = float(drawdowns.min())
    calmar    = float(ann_ret / abs(mdd)) if mdd != 0 else np.nan

    n_trades  = int((df["delta"].abs() > 0.01).sum()) \
                if "delta" in df.columns else 0
    total_cost = float(df["cost"].sum()) if "cost" in df.columns else 0.0

    bh_rets    = df["buyhold_ret"].values if "buyhold_ret" in df.columns else rets
    bh_ann     = float((1 + bh_rets).prod() ** (TRADING_DAYS / n) - 1)
    bh_excess  = bh_rets - RF_DAILY
    bh_sharpe  = float(bh_excess.mean() / bh_excess.std() * np.sqrt(TRADING_DAYS)) \
                 if bh_excess.std() > 0 else 0.0

    return {
        "ann_return"      : ann_ret,
        "sharpe"          : sharpe,
        "sortino"         : sortino,
        "max_drawdown"    : mdd,
        "calmar"          : calmar,
        "n_trades"        : n_trades,
        "total_cost_paid" : total_cost,
        "bh_ann_return"   : bh_ann,
        "bh_sharpe"       : bh_sharpe,
        "beats_bh"        : int(sharpe > bh_sharpe),
    }


# ═══════════════════════════════════════════════════════════════════════════════
# PRINT COMPARISON TABLE
# ═══════════════════════════════════════════════════════════════════════════════

def print_layer_comparison(ticker: str,
                           naive_m: dict,
                           real_m:  dict,
                           kelly_m: dict):
    """
    Side-by-side comparison of all three layers.
    The gap between naive and realistic is the most important number —
    it tells you how much friction costs you in this strategy.
    """
    print(f"\n  {'─'*68}")
    print(f"  {ticker}  —  Layer comparison")
    print(f"  {'─'*68}")
    print(f"  {'Metric':<22} {'Naive':>12} {'Realistic':>12} {'Kelly':>12}")
    print(f"  {'─'*68}")

    rows = [
        ("Ann. return",     "ann_return",      "{:>+11.2%}"),
        ("Sharpe ratio",    "sharpe",           "{:>12.3f}"),
        ("Sortino ratio",   "sortino",          "{:>12.3f}"),
        ("Max drawdown",    "max_drawdown",     "{:>+11.2%}"),
        ("Calmar ratio",    "calmar",           "{:>12.3f}"),
        ("N trades",        "n_trades",         "{:>12d}"),
        ("Total cost paid", "total_cost_paid",  "{:>+11.4f}"),
        ("B&H Sharpe",      "bh_sharpe",        "{:>12.3f}"),
        ("Beats B&H",       "beats_bh",         "{:>12d}"),
    ]

    def fmt(val, template):
        try:
            return template.format(val)
        except (ValueError, TypeError):
            return f"{'—':>12}"

    for label, key, template in rows:
        nv = naive_m.get(key, np.nan)
        rv = real_m.get(key,  np.nan)
        kv = kelly_m.get(key, np.nan)
        print(f"  {label:<22}{fmt(nv, template)}{fmt(rv, template)}{fmt(kv, template)}")

    cost_drag = (real_m.get("ann_return", 0) - naive_m.get("ann_return", 0))
    print(f"  {'─'*68}")
    print(f"  Cost drag (naive→real): {cost_drag:>+.2%} annualised")
    print(f"  {'─'*68}")


# ═══════════════════════════════════════════════════════════════════════════════
# WALK-FORWARD SUMMARY PRINTER
# ═══════════════════════════════════════════════════════════════════════════════

def print_walkforward(wf_df: pd.DataFrame, ticker: str):
    """Prints fold-by-fold walk-forward results."""
    print(f"\n  Walk-forward results — {ticker}")
    print(f"  {'Fold':<6} {'Period':<26} {'Ann Ret':>9} "
          f"{'Sharpe':>8} {'Dir Acc':>8} {'IC':>7} {'Trades':>7}")
    print(f"  {'─'*74}")

    for _, row in wf_df.iterrows():
        period = f"{row['test_start']} → {row['test_end']}"
        print(
            f"  {int(row['fold']):<6}"
            f"{period:<26}"
            f"{row['ann_return']:>+9.2%}"
            f"{row['sharpe']:>8.3f}"
            f"{row['dir_accuracy']:>8.2%}"
            f"{row['ic']:>7.4f}"
            f"{int(row['n_trades']):>7}"
        )

    print(f"  {'─'*74}")
    print(
        f"  {'Average':<32}"
        f"{wf_df['ann_return'].mean():>+9.2%}"
        f"{wf_df['sharpe'].mean():>8.3f}"
        f"{wf_df['dir_accuracy'].mean():>8.2%}"
        f"{wf_df['ic'].mean():>7.4f}"
    )

    # Consistency check
    positive_sharpe = (wf_df["sharpe"] > 0).sum()
    total_folds     = len(wf_df)
    print(f"\n  Positive Sharpe in {positive_sharpe}/{total_folds} folds "
          f"({'consistent ✓' if positive_sharpe >= total_folds * 0.7 else 'inconsistent ⚠'})")


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    import tensorflow as tf

    print("=" * 68)
    print("Stage 6 — Backtesting Engine")
    print(f"One-way cost breakdown:")
    print(f"  Bid-ask spread  : {BID_ASK_SPREAD/2:.4%}")
    print(f"  Brokerage       : {BROKERAGE:.4%}")
    print(f"  STT (sell only) : {STT_SELL:.4%}")
    print(f"  SEBI charge     : {SEBI_CHARGE:.6%}")
    print(f"  Slippage        : {SLIPPAGE:.4%}")
    print(f"  Base one-way    : {BASE_ONE_WAY_COST:.4%}")
    print("=" * 68)

    os.makedirs(BACKTEST_DIR, exist_ok=True)
    all_summary = []

    for ticker in TICKERS:
        model_path = os.path.join(MODEL_DIR, f"{ticker}_regression.keras")
        seq_path   = os.path.join(SEQ_DIR,   f"{ticker}_X_test.npy")

        if not os.path.exists(model_path) or not os.path.exists(seq_path):
            print(f"\nSKIP {ticker} — model or sequences not found.")
            continue

        print(f"\n{'═'*68}")
        print(f"  {ticker}")
        print(f"{'═'*68}")

        # Load sequences
        X_train, y_train, X_test, y_test, train_dates, test_dates = \
            load_sequences(ticker)

        # Load model and predict
        model    = tf.keras.models.load_model(model_path)
        y_pred   = model.predict(X_test, verbose=0).flatten()
        y_true   = y_test.flatten()
        dates    = pd.DatetimeIndex(test_dates)

        print(f"  Test period : {dates[0].date()} → {dates[-1].date()}")
        print(f"  Samples     : {len(y_true)}")

        # ── Layer 1: Naive ────────────────────────────────────────────────────
        print("\n  Running Layer 1 — Naive ...")
        naive_df = naive_backtest(y_true, y_pred, dates)
        naive_df.to_csv(os.path.join(BACKTEST_DIR, f"{ticker}_naive.csv"))
        naive_m = compute_metrics(
            naive_df.assign(delta=0, cost=0, buyhold_ret=y_true),
            ret_col="strategy_ret", equity_col="strategy_equity"
        )

        # ── Layer 2: Realistic ────────────────────────────────────────────────
        print("  Running Layer 2 — Realistic ...")
        real_df  = realistic_backtest(y_true, y_pred, dates)
        real_df.to_csv(os.path.join(BACKTEST_DIR, f"{ticker}_realistic.csv"))
        real_m   = compute_metrics(real_df)

        # ── Layer 3: Kelly ────────────────────────────────────────────────────
        print("  Running Layer 3 — Kelly ...")
        kelly_df = kelly_backtest(y_true, y_pred, dates)
        kelly_df.to_csv(os.path.join(BACKTEST_DIR, f"{ticker}_kelly.csv"))
        kelly_m  = compute_metrics(kelly_df)

        # ── Print comparison ──────────────────────────────────────────────────
        print_layer_comparison(ticker, naive_m, real_m, kelly_m)

        # ── Walk-forward ──────────────────────────────────────────────────────
        print("\n  Running walk-forward validation ...")
        wf_df = walk_forward_backtest(y_true, y_pred, dates)
        wf_df.to_csv(
            os.path.join(BACKTEST_DIR, f"{ticker}_walkforward.csv"), index=False
        )
        print_walkforward(wf_df, ticker)

        all_summary.append({
            "ticker"             : ticker,
            "naive_sharpe"       : naive_m["sharpe"],
            "realistic_sharpe"   : real_m["sharpe"],
            "kelly_sharpe"       : kelly_m["sharpe"],
            "realistic_mdd"      : real_m["max_drawdown"],
            "kelly_mdd"          : kelly_m["max_drawdown"],
            "cost_drag"          : real_m["ann_return"] - naive_m["ann_return"],
            "n_trades"           : real_m["n_trades"],
            "total_cost"         : real_m["total_cost_paid"],
            "wf_avg_sharpe"      : wf_df["sharpe"].mean(),
            "wf_consistency"     : float((wf_df["sharpe"] > 0).mean()),
            "beats_bh"           : real_m["beats_bh"],
        })

    # ── Cross-ticker summary ──────────────────────────────────────────────────
    if all_summary:
        summary_df = pd.DataFrame(all_summary).sort_values(
            "realistic_sharpe", ascending=False
        )
        summary_df.to_csv(
            os.path.join(BACKTEST_DIR, "summary.csv"), index=False
        )

        print("\n" + "=" * 78)
        print("FINAL SUMMARY  (sorted by Realistic Sharpe)")
        print("=" * 78)
        print(f"{'Ticker':<14}{'Naive Sh':>9}{'Real Sh':>9}{'Kelly Sh':>9}"
              f"{'Real MDD':>9}{'Cost Drag':>10}{'WF Cons':>8}{'BH':>5}")
        print("─" * 78)
        for _, row in summary_df.iterrows():
            print(
                f"{row['ticker']:<14}"
                f"{row['naive_sharpe']:>9.3f}"
                f"{row['realistic_sharpe']:>9.3f}"
                f"{row['kelly_sharpe']:>9.3f}"
                f"{row['realistic_mdd']:>9.2%}"
                f"{row['cost_drag']:>+9.2%}"
                f"{row['wf_consistency']:>8.0%}"
                f"{'✓' if row['beats_bh'] else '✗':>5}"
            )
        print("=" * 78)
        print(f"\nAll results saved to {BACKTEST_DIR}/")

    print("""
Pipeline complete. You now have:
  Stage 1  data/raw/              OHLCV CSVs
  Stage 2  data/features/         Feature-engineered CSVs
  Stage 3  data/sequences/        LSTM-ready numpy arrays
  Stage 4  models/                Trained .keras models
  Stage 5  results/               Quant evaluation metrics
  Stage 6  backtest/              Backtest equity curves + walk-forward

Resume bullet (fill in your actual numbers):
  "Designed and backtested a Bidirectional LSTM return prediction model on
  Nifty50 equities, achieving [X]% directional accuracy and Sharpe ratio of
  [Y] net of realistic NSE transaction costs (bid-ask, STT, brokerage,
  slippage), validated across [N] walk-forward folds with [Z]% fold consistency."
""")